In [1]:
import sys
sys.path.append("..")
import os
import torch
import json
import yaml
from pathlib import Path
from datasets import DatasetDict, load_from_disk
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

from transformers import AutoTokenizer, AutoModelForCausalLM
from src.data import PROJECT_ROOT
from src.dataset_cot import prepare_direct_pythia_dataset, get_cot_prompt_template, evaluate_cot_capability
from src.llm_upgrade import (
    prepare_model_for_finetune,
    train_lora_model,
    save_finetuned_model,
    wrap_for_transformer_lens,
    create_quantization_config
)

# Параметры (q)LoRA

In [2]:
EXP_ID = "exp3-2"
MODEL_NAME = "EleutherAI/pythia-1b-deduped"
VARIANT = "depth-0"
USE_SMALL = False

In [3]:
EPOCHS = 5
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1.5e-4
MAX_LENGTH = 512
SAVE_STEPS = 1000
LOGGING_STEPS = 500
EVAL_STEPS = 500

In [4]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

In [5]:
TARGET_MODULES = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

In [6]:
OUTPUT_DIR = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}")
ADAPTER_SAVE_PATH = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/lora_final")

In [7]:
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
raw_dataset = load_from_disk(str(cache_path))

In [21]:
# TRAIN_SIZE = 10000
# DEV_SIZE = 1000
SEED = 42

# Подготовка датасета

In [14]:
train_subset = raw_dataset["train"].shuffle(seed=SEED)
dev_subset = raw_dataset["dev"].shuffle(seed=SEED)

In [15]:
train_dataset = prepare_direct_pythia_dataset(train_subset)
dev_dataset   = prepare_direct_pythia_dataset(dev_subset)

In [16]:
len(train_dataset)

60557

In [17]:
len(dev_subset)

8718

In [13]:
direct_dir = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}_direct_pythia"
if dev_dataset:
    DatasetDict({"train": train_dataset, "dev": dev_dataset, "test": raw_dataset["test"]}).save_to_disk(str(direct_dir))

Saving the dataset (0/1 shards):   0%|          | 0/60557 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8718 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17347 [00:00<?, ? examples/s]

# Обучение

In [14]:
# qlora
quantization_config = create_quantization_config(use_4bit=True)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [16]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"  # Flash Attention часто нестабилен с LoRA на Windows
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [17]:
base_model = prepare_model_for_kbit_training(base_model)

In [18]:
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False
)

In [19]:
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

trainable params: 8,388,608 || all params: 1,020,170,240 || trainable%: 0.8223


In [20]:
train_config = {
    "output_dir": OUTPUT_DIR,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "logging_steps": LOGGING_STEPS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "use_wandb": False
}

In [21]:
trained_model, metrics = train_lora_model(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    config=train_config,
    max_length=MAX_LENGTH
)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
500,0.987100,0.743542
1000,0.709600,0.709547
1500,0.685600,0.691052
2000,0.674100,0.688447
2500,0.664600,0.678431
3000,0.655700,0.679510
3500,0.653900,0.673408
4000,0.649300,0.673286
4500,0.643300,0.675674
5000,0.637700,0.676566


c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\MyPythonProjects\meph

KeyboardInterrupt: 

In [22]:
save_finetuned_model(trained_model, tokenizer, ADAPTER_SAVE_PATH)

Адаптер сохранён в C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp3\lora_final


# Тест дообученной модели

In [9]:
BEST_CHECKPOINT = 6000
adapter_path = PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/checkpoint-{BEST_CHECKPOINT}"

In [10]:
hooked_model, _ = wrap_for_transformer_lens(
    MODEL_NAME,
    str(adapter_path),
    device="cuda"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer


In [18]:
eval_prefix = "{theory} {assertion} The assertion is"

In [19]:
full_test = load_from_disk(str(cache_path))["test"]

In [13]:
test_acc = evaluate_cot_capability(
    model=hooked_model,
    dataset=full_test,
    prompt_template=eval_prefix,
    batch_size=32,
    device="cuda"
)

Evaluating CoT capability:   0%|          | 0/543 [00:00<?, ?it/s]

Evaluating CoT capability: 100%|██████████| 543/543 [1:32:03<00:00, 10.17s/it]


In [22]:
results = {
    "experiment_id": EXP_ID,
    "model": MODEL_NAME,
    "variant": VARIANT,
    "best_checkpoint": adapter_path.name,
    "val_loss_at_checkpoint": 0.673169,
    "test_accuracy": test_acc,
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lora_r": LORA_R, "train_size": "full"
    }
}

In [23]:
results_path = Path(OUTPUT_DIR) / "results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

In [24]:
print(f"Experiment: {EXP_ID}")
print(f"Checkpoint: {adapter_path.name}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Results saved: {results_path}")

Experiment: exp3-2
Checkpoint: checkpoint-6000
Test Accuracy: 0.9953
Results saved: C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp3-2\results.json


# Диагностика предсказаний

In [25]:
test_examples = raw_dataset["test"].select(range(5))
eval_prefix = "{theory} {assertion} The assertion is"

In [26]:
with torch.no_grad():
    for i, ex in enumerate(test_examples):
        prompt = eval_prefix.format(theory=ex["theory"], assertion=ex["assertion"])
        tokens = hooked_model.to_tokens([prompt], prepend_bos=True).to("cuda")
        logits = hooked_model(tokens)

        # Индексы целевых токенов
        true_id = hooked_model.tokenizer.encode("True", add_special_tokens=False)[-1]
        false_id = hooked_model.tokenizer.encode("False", add_special_tokens=False)[-1]

        # Длина промпта (последний токен находится по этому индексу после добавления BOS)
        prompt_len = len(hooked_model.tokenizer.encode(prompt, add_special_tokens=False))

        logit_t = logits[0, prompt_len, true_id].item()
        logit_f = logits[0, prompt_len, false_id].item()
        pred = "True" if logit_t > logit_f else "False"
        true_label = "True" if ex["label"] else "False"

        print(f"[{i+1}] Предсказание: {pred:5s} | Логиты T/F: {logit_t:+.2f} / {logit_f:+.2f} | Истина: {true_label} | Совпало: {pred == true_label}")

[1] Предсказание: True  | Логиты T/F: +16.88 / +11.69 | Истина: True | Совпало: True
[2] Предсказание: False | Логиты T/F: +15.75 / +18.75 | Истина: False | Совпало: True
[3] Предсказание: True  | Логиты T/F: +18.25 / +15.06 | Истина: True | Совпало: True
[4] Предсказание: False | Логиты T/F: +15.50 / +19.38 | Истина: False | Совпало: True
[5] Предсказание: True  | Логиты T/F: +16.62 / +9.69 | Истина: True | Совпало: True


In [27]:
del hooked_model
torch.cuda.empty_cache()